In [34]:
# Vettorizzazione con TF-IDF con e senza stopwords


import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# Carico il dataset pulito
df = pd.read_csv("datasetpronto.csv")

# Estraggo la lista dei testi (plot puliti) e delle etichette (generi)
testi = df['Cleaned_Plot'].tolist()
etichette = df['Genre'].tolist()

# Versione CON le stopwords
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(testi)

print("Vettorizzazione CON stopwords")
print(f"Numero di testi vettorizzati: {X.shape[0]}")
print(f"Numero di caratteristiche (vocaboli unici): {X.shape[1]}")
print()

# Versione SENZA le stopwords
vectorizer_no_stopwords = TfidfVectorizer(stop_words='english')
X_no_stopwords = vectorizer_no_stopwords.fit_transform(testi)

print("Vettorizzazione SENZA stopwords")
print(f"Numero di testi vettorizzati: {X_no_stopwords.shape[0]}")
print(f"Numero di caratteristiche (vocaboli unici): {X_no_stopwords.shape[1]}")

Vettorizzazione CON stopwords
Numero di testi vettorizzati: 4398
Numero di caratteristiche (vocaboli unici): 57822

Vettorizzazione SENZA stopwords
Numero di testi vettorizzati: 4398
Numero di caratteristiche (vocaboli unici): 57519


Il numero di caratteristiche senza le stopword è minore perché semplicemente ha tolto 303 stopwords (303 parole in meno!)

In [25]:
#Vettorizzazione con CountVectorizer con e senza stopwords

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

# Carico il dataset pulito
df = pd.read_csv("datasetpronto.csv")

# Estraggo la lista dei testi e delle etichette
testi = df['Cleaned_Plot'].tolist()
etichette = df['Genre'].tolist()

# Creo un vettorizzatore CountVectorizer (con stopwords)
CVvectorizer = CountVectorizer()

# Applico il vettorizzatore ai testi
cvX = CVvectorizer.fit_transform(testi)

print("Vettorizzazione CON stopwords")
print(f"Numero di testi vettorizzati: {cvX.shape[0]}")
print(f"Numero di caratteristiche (vocaboli unici): {cvX.shape[1]}")
print()

# Versione SENZA le stopwords
CVvectorizer_no_stopwords = CountVectorizer(stop_words='english')
cvX_no_stopwords = CVvectorizer_no_stopwords.fit_transform(testi)

print("Vettorizzazione SENZA stopwords")
print(f"Numero di testi vettorizzati: {cvX_no_stopwords.shape[0]}")
print(f"Numero di caratteristiche (vocaboli unici): {cvX_no_stopwords.shape[1]}")

Vettorizzazione CON stopwords
Numero di testi vettorizzati: 4398
Numero di caratteristiche (vocaboli unici): 57822

Vettorizzazione SENZA stopwords
Numero di testi vettorizzati: 4398
Numero di caratteristiche (vocaboli unici): 57519


D'ORA IN POI INIZIA IL MODELLO.

In [ ]:
# importo GridSearchCV
from sklearn.model_selection import GridSearchCV




# importo classificatore SVC
from sklearn.svm import SVC




# instantiate classifier with default hyperparameters with kernel=rbf, C=1.0 and gamma=auto
svc=SVC()




parameters = [
    {'C': [1, 10, 100], 'kernel': ['linear']},
    {'C': [1, 10, 100], 'kernel': ['rbf']},
    {'C': [1, 10, 100], 'kernel': ['poly']}
]








grid_search = GridSearchCV(estimator = svc,  
                           param_grid = parameters,
                           scoring = 'accuracy',
                           cv = 5,
                           verbose=1)




grid_search.fit(X, etichette)

In [ ]:
# esamino il miglior modello




# best score achieved during the GridSearchCV
print('GridSearch CV best score : {:.4f}\n\n'.format(grid_search.best_score_))




# print parameters that give the best results
print('Parameters that give the best results :','\n\n', (grid_search.best_params_))




# print estimator that was chosen by the GridSearch
print('\n\nEstimator that was chosen by the search :','\n\n', (grid_search.best_estimator_))

In [ ]:
#Addestramento modello con matrice IF-IDF con stopwords

from sklearn.model_selection import train_test_split


# Divisione 80% training, 20% test
X_train, X_test, y_train, y_test = train_test_split(X, etichette, test_size=0.2, random_state=42, stratify=etichette)


from sklearn.svm import SVC


# Istanzia il modello con i parametri migliori trovati
best_model = SVC(kernel='linear', C=1)


# Addestramento
best_model.fit(X_train, y_train)


from sklearn.metrics import classification_report, accuracy_score


# Predizioni
y_pred = best_model.predict(X_test)


# Report di valutazione
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [2]:
#Addestramento modello con matrice IF-IDF senza stopwords

from sklearn.model_selection import train_test_split


# Divisione 80% training, 20% test
X_train, X_test, y_train, y_test = train_test_split(X_no_stopwords, etichette, test_size=0.2, random_state=42, stratify=etichette)


from sklearn.svm import SVC


# Istanzia il modello con i parametri migliori trovati
best_model = SVC(kernel='linear', C=1)


# Addestramento
best_model.fit(X_train, y_train)


from sklearn.metrics import classification_report, accuracy_score


# Predizioni
y_pred = best_model.predict(X_test)


# Report di valutazione
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.7238636363636364

Classification Report:
               precision    recall  f1-score   support

      action       0.78      0.80      0.79       220
      comedy       0.64      0.69      0.66       220
       drama       0.62      0.62      0.62       220
      horror       0.86      0.79      0.82       220

    accuracy                           0.72       880
   macro avg       0.73      0.72      0.73       880
weighted avg       0.73      0.72      0.73       880



In [4]:
#PROVE PER FEATURES 1/2

import numpy as np

# Recupera il vocabolario usato nel modello TF-IDF
feature_names = vectorizer_no_stopwords.get_feature_names_out()

# Ottieni i coefficienti del classificatore lineare
# Ogni riga di coef_ corrisponde a una classe
classi = best_model.classes_  # ['action', 'comedy', 'drama', 'horror']
coefficienti = best_model.coef_

# Trova l'indice associato alla classe 'horror'
indice_horror = list(classi).index('horror')

# Ottieni i coefficienti per la classe 'horror'
coefficienti_horror = coefficienti[indice_horror].toarray().flatten()

# Ordina le features per importanza (in valore assoluto o positivo/negativo)
top_n = 30  # quante parole vuoi visualizzare
top_indici = np.argsort(coefficienti_horror)[-top_n:][::-1]

print(f"Top {top_n} parole più indicative per la classe 'horror':")
for indice in top_indici:
    print(f"{feature_names[indice]}: {coefficienti_horror[indice]:.4f}")

Top 30 parole più indicative per la classe 'horror':
wedding: 1.4484
real: 1.3587
hotel: 1.3180
comedy: 1.2933
local: 1.2604
lady: 1.2511
getting: 1.2491
lucy: 1.2149
girlfriend: 1.1995
betty: 1.1953
trip: 1.1879
attractive: 1.1836
dog: 1.1427
gay: 1.1334
andrew: 1.1293
army: 1.1181
kate: 1.1021
accidentally: 1.0948
date: 1.0786
group: 1.0767
bank: 1.0749
ted: 1.0504
willie: 1.0481
girls: 1.0452
keaton: 1.0381
kevin: 1.0300
penelope: 1.0248
sylvia: 1.0197
hero: 1.0163
nora: 0.9918


In [5]:
#PROVE PER FEATURES 2/2

import numpy as np

# Recupera il vocabolario
feature_names = vectorizer_no_stopwords.get_feature_names_out()

# Ottieni i coefficienti per tutte le classi
coefficienti = best_model.coef_.toarray()
classi = best_model.classes_

# Ottieni i pesi della classe horror
indice_horror = list(classi).index('horror')
pesi_horror = coefficienti[indice_horror]

# Calcola la media dei pesi delle ALTRE classi
pesi_altre_classi = np.mean(np.delete(coefficienti, indice_horror, axis=0), axis=0)

# Calcola la differenza dei pesi: horror vs altri
delta = pesi_horror - pesi_altre_classi

# Ordina in base alla differenza
top_n = 20
indici_top = np.argsort(delta)[-top_n:][::-1]

print(f"Top {top_n} parole distintive per 'horror' rispetto alle altre classi:")
for indice in indici_top:
    print(f"{feature_names[indice]}: delta={delta[indice]:.4f}, horror_weight={pesi_horror[indice]:.4f}, others_mean={pesi_altre_classi[indice]:.4f}")

Top 20 parole distintive per 'horror' rispetto alle altre classi:
house: delta=1.7066, horror_weight=0.3272, others_mean=-1.3794
dr: delta=1.6870, horror_weight=0.1189, others_mean=-1.5680
kate: delta=1.5128, horror_weight=1.1021, others_mean=-0.4107
group: delta=1.4982, horror_weight=1.0767, others_mean=-0.4215
accidentally: delta=1.4601, horror_weight=1.0948, others_mean=-0.3654
betty: delta=1.4583, horror_weight=1.1953, others_mean=-0.2629
lucy: delta=1.4264, horror_weight=1.2149, others_mean=-0.2116
comedy: delta=1.3855, horror_weight=1.2933, others_mean=-0.0922
trip: delta=1.3815, horror_weight=1.1879, others_mean=-0.1936
hotel: delta=1.3777, horror_weight=1.3180, others_mean=-0.0596
girls: delta=1.3653, horror_weight=1.0452, others_mean=-0.3201
real: delta=1.3623, horror_weight=1.3587, others_mean=-0.0037
lou: delta=1.3320, horror_weight=0.8916, others_mean=-0.4404
wedding: delta=1.3319, horror_weight=1.4484, others_mean=0.1164
girlfriend: delta=1.3079, horror_weight=1.1995, othe

In [ ]:
#modello addestrato con la matrice creata con CountVectorizer con stopwords

from sklearn.model_selection import train_test_split


# Divisione 80% training, 20% test
X_train, X_test, y_train, y_test = train_test_split(cvX, etichette, test_size=0.2, random_state=42, stratify=etichette)


from sklearn.svm import SVC


# Istanzia il modello con i parametri migliori trovati
best_model = SVC(kernel='linear', C=1)


# Addestramento
best_model.fit(X_train, y_train)


from sklearn.metrics import classification_report, accuracy_score


# Predizioni
y_pred = best_model.predict(X_test)


# Report di valutazione
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
#modello adesstrato con la matrice creata con CountVectorizer senza stopwords

from sklearn.model_selection import train_test_split


# Divisione 80% training, 20% test
X_train, X_test, y_train, y_test = train_test_split(cvX_no_stopwords, etichette, test_size=0.2, random_state=42, stratify=etichette)


from sklearn.svm import SVC


# Istanzia il modello con i parametri migliori trovati
best_model = SVC(kernel='linear', C=1)


# Addestramento
best_model.fit(X_train, y_train)


from sklearn.metrics import classification_report, accuracy_score


# Predizioni
y_pred = best_model.predict(X_test)


# Report di valutazione
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

D'ORA IN POI FACCIAMO TEST SUL MODELLO CON IL DATASET SPORCO.

In [36]:
# Vettorizzazione con TF-IDF con e senza stopwords CON DATASET SPORCO


import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# Carico il dataset pulito
dfsporco = pd.read_csv("datasetsporcopronto.csv")

# Estraggo la lista dei testi (plot puliti) e delle etichette (generi)
testisporchi = dfsporco['Plot'].tolist()
etichette = dfsporco['Genre'].tolist()

# Versione CON le stopwords
vectorizer = TfidfVectorizer()
Xsporca = vectorizer.fit_transform(testisporchi)

print("Vettorizzazione CON stopwords")
print(f"Numero di testi vettorizzati: {Xsporca.shape[0]}")
print(f"Numero di caratteristiche (vocaboli unici): {Xsporca.shape[1]}")
print()

# Versione SENZA le stopwords
vectorizer_no_stopwords = TfidfVectorizer(stop_words='english')
Xsporca_no_stopwords = vectorizer_no_stopwords.fit_transform(testisporchi)

print("Vettorizzazione SENZA stopwords")
print(f"Numero di testi vettorizzati: {Xsporca_no_stopwords.shape[0]}")
print(f"Numero di caratteristiche (vocaboli unici): {Xsporca_no_stopwords.shape[1]}")

Vettorizzazione CON stopwords
Numero di testi vettorizzati: 4398
Numero di caratteristiche (vocaboli unici): 49652

Vettorizzazione SENZA stopwords
Numero di testi vettorizzati: 4398
Numero di caratteristiche (vocaboli unici): 49353


In [16]:
#Addestramento modello con matrice IF-IDF senza stopwords CON DATASET SPORCO (n.b.: perché riguardo al dataset sporco, vediamo solo senza stopwords)

from sklearn.model_selection import train_test_split


# Divisione 80% training, 20% test
X_train, X_test, y_train, y_test = train_test_split(Xsporca_no_stopwords, etichette, test_size=0.2, random_state=42, stratify=etichette)


from sklearn.svm import SVC


# Istanzia il modello con i parametri migliori trovati
best_model = SVC(kernel='linear', C=1)


# Addestramento
best_model.fit(X_train, y_train)


from sklearn.metrics import classification_report, accuracy_score


# Predizioni
y_pred = best_model.predict(X_test)


# Report di valutazione
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.7170454545454545

Classification Report:
               precision    recall  f1-score   support

      action       0.77      0.80      0.78       220
      comedy       0.63      0.66      0.65       220
       drama       0.62      0.61      0.62       220
      horror       0.86      0.79      0.82       220

    accuracy                           0.72       880
   macro avg       0.72      0.72      0.72       880
weighted avg       0.72      0.72      0.72       880



DALLA NOSTRA RICERCA ABBIAMO VISTO CHE:
- L'ACCURACY CON TF-IDF VECTORIZER E' MAGGIORE RISPETTO A COUNTVECTORIZER.

TF-IDF
- L'ACCURACY CON TF-IDF VECTORIZER E' MAGGIORE QUANDO RIMUOVO LE STOPWORDS RISPETTO A QUANDO LE TENGO.

COUNTVECTORIZER
- L'ACCURACY CON COUNTVECTORIZER E' MAGGIORE QUANDO RIMUOVO LE STOPWORDS RISPETTO A QUANDO LE TENGO.

DATASET SPORCO
- TF-IDF / senza stopwords / DATASET PULITO          CONFRONTO CON          TF-IDF / senza stopwords / DATASET SPORCO
         Accuracy: 0.7238636363636364                                                Accuracy: 0.7170454545454545

Quindi, pulire il dataset porta ad una maggiore accuracy.




PARTE AGGIUNTA DA ME OGGI

In [38]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

def addestra_e_valuta(X, y, nome_descrizione, C=1, kernel='linear'):

    print(f"Addestramento con: {nome_descrizione}")

    # Divisione train/test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    # Istanziazione e addestramento
    model = SVC(C=C, kernel=kernel)
    model.fit(X_train, y_train)

    # Predizione
    y_pred = model.predict(X_test)

    # Valutazione
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("\nClassification Report:\n", classification_report(y_test, y_pred))
    
    return model


In [41]:
# per far partire uno dei 4 modelli commento gli altri 3

# # TF-IDF con stopwords
# modello_tfidf_con = addestra_e_valuta(X, etichette, "TF-IDF con stopwords")

# # TF-IDF senza stopwords
# modello_tfidf_senza = addestra_e_valuta(X_no_stopwords, etichette, "TF-IDF senza stopwords")

# # CountVectorizer con stopwords
# modello_cv_con = addestra_e_valuta(cvX, etichette, "CountVectorizer con stopwords")

# # CountVectorizer senza stopwords
# modello_cv_senza = addestra_e_valuta(cvX_no_stopwords, etichette, "CountVectorizer senza stopwords")

# DATASET SPORCO

# TF-IDF senza stopwords
modello_tfidf_con_sporco = addestra_e_valuta(Xsporca_no_stopwords, etichette, "TF-IDF senza stopwords (dataset sporco)")


Addestramento con: TF-IDF senza stopwords (dataset sporco)
Accuracy: 0.7170454545454545

Classification Report:
               precision    recall  f1-score   support

      action       0.77      0.80      0.78       220
      comedy       0.63      0.66      0.65       220
       drama       0.62      0.61      0.62       220
      horror       0.86      0.79      0.82       220

    accuracy                           0.72       880
   macro avg       0.72      0.72      0.72       880
weighted avg       0.72      0.72      0.72       880



In [44]:
risultati = []

for nome, matrice in [('TF-IDF (con)', X), ('TF-IDF (senza)', X_no_stopwords),
                      ('CV (con)', cvX), ('CV (senza)', cvX_no_stopwords), ('Dataset sporco TF-IDF (senza)', Xsporca_no_stopwords)]:
    model = SVC(kernel='linear', C=1)
    X_train, X_test, y_train, y_test = train_test_split(matrice, etichette, test_size=0.2, stratify=etichette, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    risultati.append((nome, acc))

risultati


[('TF-IDF (con)', 0.7147727272727272),
 ('TF-IDF (senza)', 0.7238636363636364),
 ('CV (con)', 0.6136363636363636),
 ('CV (senza)', 0.6318181818181818),
 ('Dataset sporco TF-IDF (senza)', 0.7170454545454545)]